# GDELT News Sentiment → Databricks

Retrieves news sentiment and article volume from the GDELT DOC 2.0 API (no key required) for stocks and crypto entities. Stores raw intraday timeline observations in `bronze.gdelt_sentiment_timeline`, then builds daily aggregated features in `silver.news_sentiment_daily`. Includes exponential backoff, jitter, and per-entity failure isolation.

In [0]:
# ============================================================
# GDELT NEWS SENTIMENT -> DATABRICKS
# ============================================================
#
# Purpose:
#   Retrieve news sentiment and news volume from GDELT
#   for stocks and crypto assets.
#
# Outputs:
#   bronze.gdelt_sentiment_timeline  - raw timeline observations
#   silver.news_sentiment_daily     - daily sentiment per entity
#
# GDELT DOES NOT REQUIRE AN API KEY.
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import time
import random
import requests
import pandas as pd

from datetime import (
    datetime,
    timedelta,
    timezone
)

from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    col,
    current_timestamp,
    to_date,
    sum as spark_sum,
    avg as spark_avg,
    when
)

from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    LongType,
    TimestampType
)

from delta.tables import DeltaTable


spark = SparkSession.builder.getOrCreate()

In [0]:
# ============================================================
# 2. CONFIGURATION
# ============================================================

BASE_URL = "https://api.gdeltproject.org/api/v2/doc/doc"

BRONZE_TABLE = "bronze.gdelt_sentiment_timeline"

SILVER_TABLE = "silver.news_sentiment_daily"


# ------------------------------------------------------------
# LOAD MODE
# ------------------------------------------------------------
# BACKFILL:    Pull historical sentiment.
# INCREMENTAL: Pull recent days again and MERGE.
# ------------------------------------------------------------

LOAD_MODE = "INCREMENTAL"

BACKFILL_DAYS = 90

INCREMENTAL_LOOKBACK_DAYS = 3


# ------------------------------------------------------------
# RATE LIMIT SETTINGS
# ------------------------------------------------------------

REQUEST_TIMEOUT_SECONDS = 90

# Delay between successful API calls (GDELT is a free public service).
REQUEST_DELAY_SECONDS = 20

# Retry attempts for 429/server errors.
MAX_RETRIES = 5

# Initial exponential backoff period.
# attempt 1=30s, 2=60s, 3=120s, 4=240s, 5=failure
INITIAL_BACKOFF_SECONDS = 30

# If one entity fails, continue processing the remaining entities.
CONTINUE_ON_ENTITY_FAILURE = True

In [0]:
# ============================================================
# 3. ENTITY UNIVERSE
# ============================================================
#
# IMPORTANT: Search by company/entity name rather than ticker alone.
# Tickers such as CAT, T, META can generate unrelated news.
# ============================================================

ENTITIES = [

    {
        "entity_id": "AAPL",
        "entity_name": "Apple Inc.",
        "entity_type": "stock",
        "query": '"Apple Inc" sourcelang:english'
    },

    {
        "entity_id": "MSFT",
        "entity_name": "Microsoft",
        "entity_type": "stock",
        "query": '"Microsoft" sourcelang:english'
    },

    {
        "entity_id": "NVDA",
        "entity_name": "NVIDIA",
        "entity_type": "stock",
        "query": '"NVIDIA" sourcelang:english'
    },

    {
        "entity_id": "AMZN",
        "entity_name": "Amazon",
        "entity_type": "stock",
        "query": '"Amazon.com" sourcelang:english'
    },

    {
        "entity_id": "META",
        "entity_name": "Meta Platforms",
        "entity_type": "stock",
        "query": '"Meta Platforms" sourcelang:english'
    },

    {
        "entity_id": "JPM",
        "entity_name": "JPMorgan Chase",
        "entity_type": "stock",
        "query": '"JPMorgan Chase" sourcelang:english'
    },

    {
        "entity_id": "BAC",
        "entity_name": "Bank of America",
        "entity_type": "stock",
        "query": '"Bank of America" sourcelang:english'
    },

    {
        "entity_id": "GS",
        "entity_name": "Goldman Sachs",
        "entity_type": "stock",
        "query": '"Goldman Sachs" sourcelang:english'
    },

    {
        "entity_id": "BTC",
        "entity_name": "Bitcoin",
        "entity_type": "crypto",
        "query": '"Bitcoin" sourcelang:english'
    },

    {
        "entity_id": "ETH",
        "entity_name": "Ethereum",
        "entity_type": "crypto",
        "query": '"Ethereum" sourcelang:english'
    }

]

In [0]:
# ============================================================
# 4. DETERMINE INGESTION WINDOW
# ============================================================

END_TIME = datetime.now(timezone.utc)


if LOAD_MODE == "BACKFILL":

    START_TIME = END_TIME - timedelta(days=BACKFILL_DAYS)

elif LOAD_MODE == "INCREMENTAL":

    START_TIME = END_TIME - timedelta(days=INCREMENTAL_LOOKBACK_DAYS)

else:

    raise ValueError("LOAD_MODE must be 'BACKFILL' or 'INCREMENTAL'")


def format_gdelt_datetime(dt):
    return dt.strftime("%Y%m%d%H%M%S")


START_DATETIME = format_gdelt_datetime(START_TIME)
END_DATETIME = format_gdelt_datetime(END_TIME)


print("GDELT ingestion window:")
print(f"START: {START_TIME.isoformat()}")
print(f"END:   {END_TIME.isoformat()}")


# ============================================================
# 5. HTTP SESSION
# ============================================================

session = requests.Session()

session.headers.update({
    "Accept": "application/json",
    "User-Agent": "Databricks-Financial-Research-Sentiment-Pipeline"
})

In [0]:
# ============================================================
# 6. GDELT REQUEST FUNCTION
# ============================================================

def call_gdelt(query, mode):

    """
    Execute one GDELT DOC API request.
    Handles: HTTP 429, HTTP 5xx, network failures, malformed JSON, empty payloads.
    Returns: dict on success, None after retry exhaustion.
    """

    params = {
        "query": query,
        "mode": mode,
        "format": "json",
        "startdatetime": START_DATETIME,
        "enddatetime": END_DATETIME
    }


    for attempt in range(1, MAX_RETRIES + 1):

        try:

            response = session.get(
                BASE_URL,
                params=params,
                timeout=REQUEST_TIMEOUT_SECONDS
            )


            # =================================================
            # 429 RATE LIMIT
            # =================================================

            if response.status_code == 429:

                retry_after = response.headers.get("Retry-After")

                if retry_after:
                    try:
                        wait_seconds = float(retry_after)
                    except ValueError:
                        wait_seconds = INITIAL_BACKOFF_SECONDS * (2 ** (attempt - 1))
                else:
                    wait_seconds = INITIAL_BACKOFF_SECONDS * (2 ** (attempt - 1))

                # Add jitter so repeated clients don't retry simultaneously.
                wait_seconds += random.uniform(2, 8)

                print(f"GDELT HTTP 429 | attempt {attempt}/{MAX_RETRIES}")

                if attempt == MAX_RETRIES:
                    print("Maximum 429 retries reached.")
                    return None

                print("Rate limited. Backing off before retry.")
                time.sleep(wait_seconds)
                continue


            # =================================================
            # SERVER ERROR
            # =================================================

            if response.status_code >= 500:

                print(f"GDELT HTTP {response.status_code} | attempt {attempt}/{MAX_RETRIES}")

                if attempt == MAX_RETRIES:
                    return None

                wait_seconds = 15 * attempt + random.uniform(1, 5)
                time.sleep(wait_seconds)
                continue


            # =================================================
            # OTHER HTTP ERRORS
            # =================================================

            response.raise_for_status()


            # =================================================
            # PARSE JSON
            # =================================================

            try:
                payload = response.json()

            except ValueError:
                print("GDELT returned non-JSON content.")
                print(response.text[:300])

                if attempt == MAX_RETRIES:
                    return None

                time.sleep(15 * attempt)
                continue


            # =================================================
            # EMPTY RESPONSE
            # =================================================

            if payload is None:
                print("GDELT returned an empty payload.")
                return None

            if not isinstance(payload, dict):
                print("Unexpected GDELT response type.")
                return None

            return payload


        except (
            requests.exceptions.Timeout,
            requests.exceptions.ConnectionError
        ) as error:

            print(f"Network error | attempt {attempt}/{MAX_RETRIES}")
            print(str(error))

            if attempt == MAX_RETRIES:
                return None

            time.sleep(15 * attempt)


        except requests.exceptions.RequestException as error:

            print("HTTP request failed:")
            print(str(error))
            return None

    return None

In [0]:
# ============================================================
# 7. TIMELINE PARSER
# ============================================================

def parse_timeline(payload, value_column):

    """
    Convert a GDELT timeline payload into a Pandas DataFrame.
    """

    output_columns = [
        "observation_timestamp",
        value_column,
        "all_articles"
    ]


    # --------------------------------------------
    # PROTECT AGAINST NONE
    # --------------------------------------------

    if payload is None:
        return pd.DataFrame(columns=output_columns)


    timeline = payload.get("timeline")

    if not timeline:
        return pd.DataFrame(columns=output_columns)


    # GDELT timelines can contain one or multiple series.
    series = timeline[0]

    observations = series.get("data", [])

    if not observations:
        return pd.DataFrame(columns=output_columns)


    rows = []

    for observation in observations:

        timestamp = pd.to_datetime(
            observation.get("date"),
            utc=True,
            errors="coerce"
        )

        value = pd.to_numeric(
            observation.get("value"),
            errors="coerce"
        )

        norm = pd.to_numeric(
            observation.get("norm"),
            errors="coerce"
        )

        rows.append({
            "observation_timestamp": timestamp,
            value_column: value,
            "all_articles": norm
        })


    result = pd.DataFrame(rows)

    result = result.dropna(subset=["observation_timestamp"])

    return result


# ============================================================
# 8. FETCH ONE ENTITY
# ============================================================

def fetch_entity(entity):

    entity_id = entity["entity_id"]
    entity_name = entity["entity_name"]
    entity_type = entity["entity_type"]
    query = entity["query"]

    print(f"\nFetching sentiment for {entity_id} - {entity_name}")


    # ========================================================
    # REQUEST TONE
    # ========================================================

    print("Requesting news tone...")

    tone_payload = call_gdelt(query=query, mode="timelinetone")

    if tone_payload is None:
        print(f"{entity_id}: tone request failed.")
        return None

    tone_df = parse_timeline(payload=tone_payload, value_column="average_tone")

    if tone_df.empty:
        print(f"{entity_id}: no tone observations.")
        return None


    # ========================================================
    # WAIT BEFORE SECOND REQUEST
    # ========================================================

    time.sleep(REQUEST_DELAY_SECONDS + random.uniform(1, 5))


    # ========================================================
    # REQUEST ARTICLE VOLUME
    # ========================================================

    print("Requesting article volume...")

    volume_payload = call_gdelt(query=query, mode="timelinevolraw")


    # Volume failure should NOT discard otherwise valid sentiment data.

    if volume_payload is None:

        print(f"{entity_id}: volume request failed. Keeping tone data.")
        result = tone_df.copy()
        result["article_count"] = None

    else:

        volume_df = parse_timeline(payload=volume_payload, value_column="article_count")

        if volume_df.empty:

            result = tone_df.copy()
            result["article_count"] = None

        else:

            volume_columns = ["observation_timestamp", "article_count"]

            # Use all_articles from volume because TimelineVolRaw returns
            # the normalization count.
            if "all_articles" in volume_df.columns:
                volume_columns.append("all_articles")

            # Remove norm from tone before joining.
            tone_clean = tone_df.drop(
                columns=["all_articles"],
                errors="ignore"
            )

            result = tone_clean.merge(
                volume_df[volume_columns],
                on="observation_timestamp",
                how="left"
            )


    # ========================================================
    # ENSURE REQUIRED COLUMNS EXIST
    # ========================================================

    if "all_articles" not in result.columns:
        result["all_articles"] = None

    if "article_count" not in result.columns:
        result["article_count"] = None


    # ========================================================
    # ADD ENTITY METADATA
    # ========================================================

    result["entity_id"] = entity_id
    result["entity_name"] = entity_name
    result["entity_type"] = entity_type
    result["search_query"] = query
    result["data_source"] = "GDELT DOC 2.0"

    print(f"{entity_id}: {len(result):,} timeline observations.")

    return result

In [0]:
# ============================================================
# 9. INGEST ALL ENTITIES
# ============================================================

frames = []
successful_entities = []
failed_entities = []


for index, entity in enumerate(ENTITIES, start=1):

    entity_id = entity["entity_id"]

    print("\n====================================")
    print(f"[{index}/{len(ENTITIES)}] {entity_id}")
    print("====================================")

    try:

        entity_df = fetch_entity(entity)

        if entity_df is not None and not entity_df.empty:
            frames.append(entity_df)
            successful_entities.append(entity_id)
        else:
            print(f"{entity_id}: no usable data.")
            failed_entities.append(entity_id)

    except Exception as error:

        print(f"{entity_id}: unexpected failure.")
        print(repr(error))
        failed_entities.append(entity_id)

        if not CONTINUE_ON_ENTITY_FAILURE:
            raise

    # ========================================================
    # WAIT BEFORE NEXT ENTITY
    # ========================================================

    if index < len(ENTITIES):
        print("Waiting before next entity...")
        time.sleep(REQUEST_DELAY_SECONDS + random.uniform(2, 6))


# ============================================================
# 10. STOP IF NOTHING WORKED
# ============================================================

if not frames:

    raise RuntimeError(
        "No GDELT sentiment data was retrieved.\n\n"
        "GDELT may currently be rate-limiting the Databricks "
        "network/IP address.\n\n"
        "No Delta tables were modified."
    )

In [0]:
# ============================================================
# 11. COMBINE PANDAS DATA
# ============================================================

combined_pdf = pd.concat(frames, ignore_index=True)

print("\n====================================")
print("DOWNLOAD SUMMARY")
print("====================================")
print(f"Successful entities: {len(successful_entities)}")
print(f"Failed entities: {len(failed_entities)}")
print(f"Rows downloaded: {len(combined_pdf):,}")

if failed_entities:
    print("Failed entity IDs:")
    print(failed_entities)


# ============================================================
# 12. CLEAN NUMERIC FIELDS
# ============================================================

for field in ["average_tone", "article_count", "all_articles"]:
    combined_pdf[field] = pd.to_numeric(combined_pdf[field], errors="coerce")


# ============================================================
# 13. CLEAN TIMESTAMP
# ============================================================

combined_pdf["observation_timestamp"] = pd.to_datetime(
    combined_pdf["observation_timestamp"],
    utc=True,
    errors="coerce"
)

combined_pdf = combined_pdf.dropna(
    subset=["entity_id", "observation_timestamp", "average_tone"]
)


# ============================================================
# 14. REMOVE DUPLICATES
# ============================================================

combined_pdf = (
    combined_pdf
    .sort_values("observation_timestamp")
    .drop_duplicates(
        subset=["entity_id", "observation_timestamp"],
        keep="last"
    )
)


# ============================================================
# 15. CONVERT UTC TIMESTAMP FOR SPARK
# ============================================================

combined_pdf["observation_timestamp"] = (
    combined_pdf["observation_timestamp"]
    .dt.tz_convert("UTC")
    .dt.tz_localize(None)
)


# ============================================================
# 16. REPLACE NAN WITH NONE
# ============================================================

combined_pdf = combined_pdf.where(pd.notnull(combined_pdf), None)

In [0]:
# ============================================================
# 17. SPARK SCHEMA
# ============================================================

bronze_schema = StructType([

    StructField("entity_id", StringType(), False),
    StructField("entity_name", StringType(), True),
    StructField("entity_type", StringType(), True),
    StructField("observation_timestamp", TimestampType(), False),
    StructField("average_tone", DoubleType(), True),
    StructField("article_count", DoubleType(), True),
    StructField("all_articles", DoubleType(), True),
    StructField("search_query", StringType(), True),
    StructField("data_source", StringType(), False)

])


# ============================================================
# 18. CREATE SPARK DATAFRAME
# ============================================================

spark_df = spark.createDataFrame(
    combined_pdf[[
        "entity_id",
        "entity_name",
        "entity_type",
        "observation_timestamp",
        "average_tone",
        "article_count",
        "all_articles",
        "search_query",
        "data_source"
    ]],
    schema=bronze_schema
)


bronze_df = (
    spark_df
    .withColumn("article_count", col("article_count").cast("long"))
    .withColumn("all_articles", col("all_articles").cast("long"))
    .withColumn("ingestion_timestamp", current_timestamp())
)

In [0]:
# ============================================================
# 19. CREATE DATABASE SCHEMAS
# ============================================================

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")


# ============================================================
# 20. CREATE BRONZE TABLE
# ============================================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
    entity_id STRING,
    entity_name STRING,
    entity_type STRING,
    observation_timestamp TIMESTAMP,
    average_tone DOUBLE,
    article_count BIGINT,
    all_articles BIGINT,
    search_query STRING,
    data_source STRING,
    ingestion_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'GDELT news sentiment and article volume timeline'
""")


# ============================================================
# 21. BRONZE DELTA MERGE
# ============================================================

bronze_target = DeltaTable.forName(spark, BRONZE_TABLE)

(
    bronze_target
    .alias("target")
    .merge(
        bronze_df.alias("source"),
        """
        target.entity_id = source.entity_id
        AND
        target.observation_timestamp = source.observation_timestamp
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Bronze sentiment MERGE complete.")

In [0]:
# ============================================================
# 22. BUILD DAILY SILVER DATA
# ============================================================
#
# We create daily observations because stock data, crypto data,
# and risk models operate primarily at daily frequency.
#
# Daily tone is article-volume weighted whenever article counts
# are available; otherwise falls back to simple average.
# ============================================================

daily_base_df = (
    bronze_df
    .withColumn(
        "sentiment_date",
        to_date(col("observation_timestamp"))
    )
)


# ============================================================
# 23. DAILY AGGREGATION
# ============================================================

daily_df = (
    daily_base_df
    .groupBy(
        "entity_id",
        "entity_name",
        "entity_type",
        "sentiment_date"
    )
    .agg(
        spark_sum("article_count").alias("article_count"),
        spark_sum("all_articles").alias("all_articles"),
        spark_sum(
            col("average_tone") * col("article_count")
        ).alias("weighted_tone_numerator"),
        spark_avg("average_tone").alias("simple_average_tone")
    )
)


# ============================================================
# 24. CALCULATE DAILY TONE
# ============================================================

daily_df = (
    daily_df
    .withColumn(
        "average_tone",
        when(
            col("article_count") > 0,
            col("weighted_tone_numerator") / col("article_count")
        )
        .otherwise(col("simple_average_tone"))
    )
    .withColumn(
        "coverage_share",
        when(
            col("all_articles") > 0,
            col("article_count") / col("all_articles")
        )
    )
    .drop("weighted_tone_numerator", "simple_average_tone")
    .withColumn("ingestion_timestamp", current_timestamp())
)

In [0]:
# ============================================================
# 25. CREATE SILVER TABLE
# ============================================================

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_TABLE} (
    entity_id STRING,
    entity_name STRING,
    entity_type STRING,
    sentiment_date DATE,
    average_tone DOUBLE,
    article_count BIGINT,
    all_articles BIGINT,
    coverage_share DOUBLE,
    ingestion_timestamp TIMESTAMP
)
USING DELTA
COMMENT 'Daily GDELT news sentiment by financial entity'
""")


# ============================================================
# 26. SILVER MERGE
# ============================================================

silver_target = DeltaTable.forName(spark, SILVER_TABLE)

(
    silver_target
    .alias("target")
    .merge(
        daily_df.alias("source"),
        """
        target.entity_id = source.entity_id
        AND
        target.sentiment_date = source.sentiment_date
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Silver daily sentiment MERGE complete.")

In [0]:
# ============================================================
# 27. VALIDATE BRONZE
# ============================================================

print("\nBRONZE SUMMARY")

display(
    spark.sql(f"""
    SELECT
        entity_id,
        COUNT(*) AS observations,
        MIN(observation_timestamp) AS earliest_timestamp,
        MAX(observation_timestamp) AS latest_timestamp,
        ROUND(AVG(average_tone), 4) AS average_tone
    FROM {BRONZE_TABLE}
    GROUP BY entity_id
    ORDER BY entity_id
    """)
)


# ============================================================
# 28. DISPLAY DAILY SENTIMENT
# ============================================================

print("\nLATEST DAILY SENTIMENT")

display(
    spark.sql(f"""
    SELECT
        entity_id,
        entity_name,
        entity_type,
        sentiment_date,
        ROUND(average_tone, 4) AS average_tone,
        article_count,
        ROUND(coverage_share, 6) AS coverage_share
    FROM {SILVER_TABLE}
    ORDER BY sentiment_date DESC, entity_id
    LIMIT 100
    """)
)


# ============================================================
# 29. FINAL JOB SUMMARY
# ============================================================

print("\n========================================")
print("GDELT SENTIMENT INGESTION COMPLETE")
print("========================================")
print(f"Load mode: {LOAD_MODE}")
print(f"Requested entities: {len(ENTITIES)}")
print(f"Successful entities: {len(successful_entities)}")
print(f"Failed entities: {len(failed_entities)}")
print(f"Bronze table: {BRONZE_TABLE}")
print(f"Silver table: {SILVER_TABLE}")

if failed_entities:
    print("\nFailed entities:")
    print(failed_entities)

print("========================================")